<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_3/lessons/lesson_30_redis_overview/note_lesson_30_redis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ Урок 30 — Redis: лічильники, кеш, черги й рейтинг диспетчерської

| Крок | Що робимо |
|---|---|
| 0 | запускаємо Redis і підключаємось з Python |
| 1 | лічильник з часом життя (вправа 1) |
| 2 | list — черга SMS (вправа 2) |
| 3 | hash — картка кур'єра (вправа 3) |
| 4 | set — унікальні клієнти (вправа 4) |
| 5 | sorted set — рейтинг (вправа 5) |
| 6 | cache-aside (вправа 6) |
| 7 | rate limit (вправа 7) |
| 8 | знайди помилку: `GET` + `SET` (вправа 8) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам. Теорія й схеми — у книзі: [Урок 30](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m3/lesson_30/).

---
## 0. Redis і підключення

- **Google Colab:** клітинка нижче встановить Redis у віртуальну машину Colab і запустить сервер.
- **Свій комп'ютер:** запусти Redis заздалегідь (`docker run -p 6379:6379 -d redis:7` або `sudo apt install redis-server`). Інша адреса — змінна середовища `REDIS_URL`.

Ноутбук працює в **окремій базі Redis з номером 15** і очищає лише її — твої інші дані в Redis (база 0) не постраждають.

In [ ]:
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run("apt-get -qq update && apt-get -qq install -y redis-server > /dev/null", shell=True, check=True)
    subprocess.run("redis-server --daemonize yes", shell=True, check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "redis"], check=True)
print("готово")

In [ ]:
import json
import threading
import time

import redis

REDIS_URL = os.environ.get("REDIS_URL", "redis://localhost:6379/15")
r = redis.Redis.from_url(REDIS_URL, decode_responses=True)
print(r.ping())
r.flushdb()          # чиста база 15 при кожному запуску
print("ключів:", r.dbsize())

---
## 1. Лічильник з часом життя

**🔮 Прогноз:** що поверне `r.ttl(key)` для ключа без часу життя? А для ключа, якого немає?

<details>
<summary>Відповідь</summary>

`-1` — ключ живе вічно; `-2` — ключа немає.

</details>

In [ ]:
r.set("courier:1:name", "Оксана")
print(r.ttl("courier:1:name"), r.ttl("no:such:key"))

### Вправа 1. `count_view(page)`

Лічильник переглядів сторінки меню: ключ `views:<page>`, кожен виклик додає 1 **атомарно** і повертає нове значення. Ключ має жити **60 секунд від першого перегляду** (TTL ставимо лише тоді, коли лічильник щойно став 1).

In [ ]:
def count_view(page):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    key = f"views:{page}"
    count = r.incr(key)
    if count == 1:
        r.expire(key, 60)
    return count
    # END SOLUTION


assert [count_view("pizza") for _ in range(3)] == [1, 2, 3]
assert count_view("sushi") == 1
assert 0 < r.ttl("views:pizza") <= 60
print("✅ Вправа 1 пройдена")

---
## 2. List: черга SMS

`RPUSH` — у хвіст, `LPOP` — з голови: черга FIFO, як `Queue` з уроку 28.

### Вправа 2. `enqueue_sms` / `next_sms`

- `enqueue_sms(text)` — додати текст у хвіст списку `queue:sms`;
- `next_sms()` — забрати **найстаріше** повідомлення; якщо черга порожня — `None`.

In [ ]:
def enqueue_sms(text):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    r.rpush("queue:sms", text)
    # END SOLUTION


def next_sms():
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return r.lpop("queue:sms")
    # END SOLUTION


for text in ["#101 прийнято", "#102 кур'єр їде", "#103 доставлено"]:
    enqueue_sms(text)
assert r.llen("queue:sms") == 3
assert [next_sms(), next_sms()] == ["#101 прийнято", "#102 кур'єр їде"]
assert next_sms() == "#103 доставлено"
assert next_sms() is None
print("✅ Вправа 2 пройдена")

---
## 3. Hash: картка кур'єра

### Вправа 3. `save_courier` і `add_delivery`

- `save_courier(courier_id, name, district)` — hash `courier:<id>` з полями `name`, `district`, `delivered` = 0;
- `add_delivery(courier_id)` — **атомарно** збільшити `delivered` на 1 і повернути нове значення.

In [ ]:
def save_courier(courier_id, name, district):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    r.hset(f"courier:{courier_id}", mapping={"name": name, "district": district, "delivered": 0})
    # END SOLUTION


def add_delivery(courier_id):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return r.hincrby(f"courier:{courier_id}", "delivered", 1)
    # END SOLUTION


save_courier(7, "Тарас", "Центр")
assert add_delivery(7) == 1 and add_delivery(7) == 2
print(r.hgetall("courier:7"))
assert r.hgetall("courier:7") == {"name": "Тарас", "district": "Центр", "delivered": "2"}
print("✅ Вправа 3 пройдена")

---
## 4. Set: унікальні клієнти

**🔮 Прогноз:** `SADD day "Анна" "Богдан" "Анна"` — що поверне команда?

<details>
<summary>Відповідь</summary>

`2`: кількість **нових** елементів. Друга «Анна» вже є в множині.

</details>

### Вправа 4. Постійні клієнти

- `record_customers(day, names)` — додати імена у множину `customers:<day>` і повернути, **скільки унікальних** клієнтів тепер за цей день;
- `loyal(day1, day2)` — **відсортований** список клієнтів, які замовляли в обидва дні.

In [ ]:
def record_customers(day, names):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    r.sadd(f"customers:{day}", *names)
    return r.scard(f"customers:{day}")
    # END SOLUTION


def loyal(day1, day2):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return sorted(r.sinter(f"customers:{day1}", f"customers:{day2}"))
    # END SOLUTION


assert record_customers("mon", ["Анна", "Богдан", "Віра", "Анна"]) == 3
assert record_customers("tue", ["Анна", "Галина", "Віра"]) == 3
assert loyal("mon", "tue") == ["Анна", "Віра"]
print("✅ Вправа 4 пройдена")

---
## 5. Sorted set: рейтинг кур'єрів

### Вправа 5. `add_sum`, `top`, `place`

Відсортована множина `rating:week`:

- `add_sum(courier, amount)` — додати суму до балу кур'єра;
- `top(n)` — список `(кур'єр, сума)` від найбільшої суми (суми — `float`, як їх повертає redis-py);
- `place(courier)` — місце в рейтингу **з одиниці**.

In [ ]:
def add_sum(courier, amount):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    r.zincrby("rating:week", amount, courier)
    # END SOLUTION


def top(n):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return r.zrevrange("rating:week", 0, n - 1, withscores=True)
    # END SOLUTION


def place(courier):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return r.zrevrank("rating:week", courier) + 1
    # END SOLUTION


for courier, amount in [("Оксана", 540), ("Тарас", 1250), ("Оксана", 980), ("Ігор", 760)]:
    add_sum(courier, amount)
print(top(3))
assert top(2) == [("Оксана", 1520.0), ("Тарас", 1250.0)]
assert place("Ігор") == 3
add_sum("Тарас", 300)
assert place("Тарас") == 1
print("✅ Вправа 5 пройдена")

---
## 6. Cache-aside

Спершу Redis; немає (**miss**) — рахуємо «в базі» і кладемо в Redis з TTL; є (**hit**) — віддаємо одразу.

### Вправа 6. `district_report(district)`

`slow_report(district)` — «повільний запит до PostgreSQL» (рахує свої виклики). Напиши `district_report`, що кешує результат у ключі `cache:report:<district>` на **120 секунд**. Словник зберігай як JSON (`json.dumps` / `json.loads`).

In [ ]:
calls = []


def slow_report(district):
    calls.append(district)
    time.sleep(0.2)
    return {"district": district, "revenue": {"Поділ": 2730.0, "Оболонь": 2010.0}.get(district, 0.0)}


def district_report(district):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    key = f"cache:report:{district}"
    cached = r.get(key)
    if cached is not None:
        return json.loads(cached)
    report = slow_report(district)
    r.set(key, json.dumps(report, ensure_ascii=False), ex=120)
    return report
    # END SOLUTION


for district in ["Поділ", "Поділ", "Оболонь", "Поділ"]:
    print(district_report(district))
assert calls == ["Поділ", "Оболонь"]
assert 0 < r.ttl("cache:report:Поділ") <= 120
print("✅ Вправа 6 пройдена: база працювала", len(calls), "рази на 4 запити")

---
## 7. Rate limit

### Вправа 7. `allow(action, user, minute, limit)`

Не більше `limit` дій за хвилину: ключ `rate:<action>:<user>:<minute>`, `INCR`, TTL 60 секунд на першій дії. Повертає `True`, якщо дію дозволено.

In [ ]:
def allow(action, user, minute, limit):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    key = f"rate:{action}:{user}:{minute}"
    count = r.incr(key)
    if count == 1:
        r.expire(key, 60)
    return count <= limit
    # END SOLUTION


promo = [allow("promo", "+380501112233", "12:05", 5) for _ in range(6)]
print(promo)
assert promo == [True] * 5 + [False]
assert allow("order", "+380501112233", "12:05", 3) is True          # інша дія — свій лічильник
assert allow("promo", "+380501112233", "12:06", 5) is True          # нова хвилина
assert 0 < r.ttl("rate:promo:+380501112233:12:05") <= 60
print("✅ Вправа 7 пройдена")

---
## 8. Знайди помилку

### Вправа 8. Лічильник, що губить замовлення

Колега рахує замовлення так:

```python
def count_order():
    current = int(r.get("orders:today") or 0)
    r.set("orders:today", current + 1)
```

Чотири «сервери» (потоки) приймають по 500 замовлень — і частина губиться. Перепиши `count_order`, щоб усі 2000 врахувалися.

In [ ]:
def count_order():
    # YOUR CODE HERE
    # BEGIN SOLUTION
    r.incr("orders:today")
    # END SOLUTION


r.delete("orders:today")
threads = [threading.Thread(target=lambda: [count_order() for _ in range(500)]) for _ in range(4)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print("орієнтовно 2000, маємо:", r.get("orders:today"))
assert r.get("orders:today") == "2000"
print("✅ Вправа 8 пройдена")

---
## Самоперевірка

1. Що класти в Redis, а що — лише в PostgreSQL?
2. Чому `INCR` безпечний з кількох серверів, а `GET` + `SET` — ні?
3. Яку структуру Redis взяти для черги, картки, унікальних значень, рейтингу?
4. Чому кеш завжди має TTL?

<details>
<summary>Відповіді</summary>

1. Redis — те, що можна відновити або що живе недовго: кеш, лічильники, черги, сесії. PostgreSQL — джерело правди: замовлення, гроші.
2. Redis виконує команди по одній; `INCR` — одна команда. Між `GET` і `SET` встигають команди інших клієнтів.
3. List, hash, set, sorted set.
4. Кеш — копія, що застаріває; TTL гарантує, що вона колись оновиться.

</details>

In [ ]:
r.flushdb()
r.close()

## Далі

- Книга: [Урок 30](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m3/lesson_30/) — Pub/Sub, персистентність, `maxmemory-policy`, протокол RESP.
- Документація: [Redis data types](https://redis.io/docs/latest/develop/data-types/), [redis-py](https://redis.readthedocs.io/en/stable/).
- **Урок 31** — HTTP: `requests`, `httpx`, `aiohttp`.